<a href="https://colab.research.google.com/github/MarioLara-ui/PROYECTO-FINAL/blob/main/Borrador_trabajo_final_FP1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def agregar(inventario, nombre, cantidad, costo_lote, precio_venta):

    costo_unitario = costo_lote/cantidad

    if nombre in inventario:
        inventario[nombre]["cantidad"] += cantidad
        inventario[nombre]["costo_lote"] += costo_lote
        inventario[nombre]["costo_unitario"] = inventario[nombre]["costo_lote"] / inventario[nombre]["cantidad"]
        inventario[nombre]["precio_venta"] = precio_venta
    else:
        inventario[nombre] = {
            "cantidad": cantidad,
            "costo_unitario": costo_unitario,
            "costo_lote": costo_lote,
            "precio_venta": precio_venta
        }

def consultar(inventario, nombre):
    if nombre in inventario:
        return inventario[nombre]["cantidad"]
    return None

def verificar_stock(cantidad):
    stock_minimo = 5

    if cantidad == 0:
        print("")
        print("¡ALERTA! MERCADERÍA AGOTADA")
        print("¡Solicitar reposición de mercadería inmediatamente!")
        print("4")
    elif cantidad <= stock_minimo:
        print("")
        print("¡ADVERTENCIA! QUEDA POCA MERCADERÍA")
        print("Se recomienda solicitar más unidades")
        print("")


def registrar_venta(inventario, ventas, nombre, cantidad, dia, ingreso_total, costo_total):
    if nombre not in inventario:
        print("La mercadería no existe")
        return ingreso_total, costo_total

    if cantidad <= 0:
        print("La cantidad debe ser mayor que cero")
        return ingreso_total, costo_total

    if cantidad > inventario[nombre]["cantidad"]:
        print("No hay suficientes unidades disponibles")
        return ingreso_total, costo_total

    costo_unitario = inventario[nombre]["costo_unitario"]
    precio_venta = inventario[nombre]["precio_venta"]

    ingreso_venta = cantidad * precio_venta
    costo_venta = cantidad * costo_unitario
    gananciab_venta = ingreso_venta - costo_venta

    ingreso_total += ingreso_venta
    costo_total += costo_venta

    inventario[nombre]["cantidad"] -= cantidad
    inventario[nombre]["costo_lote"] = inventario[nombre]["cantidad"] * costo_unitario

    ventas.append({
        "mercaderia": nombre,
        "cantidad": cantidad,
        "costo_unitario": costo_unitario,
        "precio_venta": precio_venta,
        "total_venta": ingreso_venta,
        "costo_total": costo_venta,
        "ganancia_bruta": gananciab_venta,
        "dia": dia
    })


    print("")
    print("Venta registrada correctamente")
    print("")
    print("Total a cobrar: S/", round(ingreso_venta, 2))




    verificar_stock(inventario[nombre]["cantidad"])

    return ingreso_total, costo_total


def ingresos_dia(ventas, dia):
    total = 0

    for venta in ventas:
        if venta["dia"] == dia:
            total += venta["total_venta"]

    return total

def mercaderia_mas_vendida(ventas):
    cantidades = {}

    for venta in ventas:
        nombre = venta["mercaderia"]
        cantidad = venta["cantidad"]

        if nombre in cantidades:
            cantidades[nombre] += cantidad
        else:
            cantidades[nombre] = cantidad

    if len(cantidades) == 0:
        return None

    nombre_mayor = ""
    cantidad_mayor = 0

    for nombre in cantidades:
        if cantidades[nombre] > cantidad_mayor:
            cantidad_mayor = cantidades[nombre]
            nombre_mayor = nombre

    return nombre_mayor, cantidad_mayor



# PROGRAMA PRINCIPAL

inventario = {}
ventas = []
ingreso_total = 0
costo_total = 0

while True:
    print("\n========== SISTEMA DE INVENTARIO ==========")
    print("1. Ingreso de mercadería")
    print("2. Consulta de stock por producto")
    print("3. Registrar venta")
    print("4. Consultar ventas por fecha")
    print("5. EERR simplificado")
    print("6. Producto más vendido")
    print("7. Historial de ventas")
    print("8. Reporte general de inventario")
    print("9. Salir")

    opcion = input("\nSeleccione una opción: ").strip()

    # OPCIÓN 1: INGRESO DE MERCADERÍA
    if opcion == "1":
        nombre = input("Nombre del producto: ").strip()

        if nombre == "":
            print("El nombre no puede estar vacío")
            continue

        try:
            cantidad = int(input("Cantidad: "))
            costo_lote = float(input("Costo lote: S/ "))
            precio_venta = float(input("Precio de venta unitario: S/ "))

            if cantidad <= 0 or costo_lote < 0 or precio_venta < 0:
                print("Ingrese valores válidos")
                continue

            costo_unitario = costo_lote/cantidad

            agregar(
                inventario,
                nombre,
                cantidad,
                costo_lote,
                precio_venta
            )

            print("")
            print("Producto registrado correctamente")
            verificar_stock(inventario[nombre]["cantidad"])

        except ValueError:
            print("Error: ingrese cantidades y precios numéricos")

    # OPCIÓN 2: CONSULTA DE STOCK POR PRODUCTO
    elif opcion == "2":
        nombre = input("Nombre del producto: ").strip()
        cantidad = consultar(inventario, nombre)

        if cantidad is None:
            print("El producto no existe en el registro de inventario")
        else:
            print("")
            print("Producto:", nombre)
            print("Stock disponible:", cantidad)
            verificar_stock(cantidad)

    # OPCIÓN 3: REGISTRAR VENTA
    elif opcion == "3":
        print("")
        nombre = input("Producto ").strip()

        try:
            cantidad = int(input("Cantidad: "))
            dia = input(
                "Fecha (ejemplo: 09/10/2026): "
            ).strip()

            if dia == "":
                print("Debe ingresar el día de la venta")
                continue

            ingreso_total, costo_total = registrar_venta(
                inventario,
                ventas,
                nombre,
                cantidad,
                dia,
                ingreso_total,
                costo_total
            )

        except ValueError:
            print("Error: ingrese una cantidad entera válida")

    # OPCIÓN 4: Ingreso por ventas del día
    elif opcion == "4":
        print("")
        dia = input("Ingrese el día que desea consultar: ").strip()
        total = ingresos_dia(ventas, dia)

        print("Ingresos del día", dia, ": S/", round(total, 2))

    # OPCIÓN 5: EERR simplificado
    elif opcion == "5":

        #GASTOS FIJOS DEL NEGOCIO
        pago_alquiler = 1500
        pago_servicios = 410 #Gastos de agua y luz
        salario_personal = 1300 #Salario de 1 empleado
        gasto_fijo = pago_alquiler + pago_servicios + salario_personal

        ganancia_bruta = ingreso_total - costo_total
        ganancia_neta = ganancia_bruta - gasto_fijo

        print("")
        print("EERR Simplificado")
        print("")
        print("Ingresos:           s/", ingreso_total)
        print("CMV:              - s/", costo_total)
        print("___________________________")
        print("Ganancia bruta:     s/", ganancia_bruta)
        print("Gastos fijos:     - s/", gasto_fijo)
        print("___________________________")
        print("Ganancia neta:      s/", ganancia_neta)

    # OPCIÓN 6: PRODUCTO MÁS VENDIDO
    elif opcion == "6":
        resultado = mercaderia_mas_vendida(ventas)

        if resultado is None:
            print("Todavía no hay ventas")
        else:
            nombre, cantidad = resultado
            print("")
            print("Producto más vendido:", nombre)
            print("Cantidad:", cantidad)

    # OPCIÓN 7: HISTORIAL DE VENTAS
    elif opcion == "7":
        print("")
        if len(ventas) == 0:
            print("Todavía no hay ventas")
        else:
            print("")
            print("HISTORIAL DE VENTAS")
            print("")

            numero = 1
            for venta in ventas:
                print(numero,
                      ".",
                      venta["dia"],
                      "| Producto:",
                      venta["mercaderia"],
                      "| Cantidad:",
                      venta["cantidad"],
                      "| Ingreso: S/",
                      round(venta["total_venta"], 2)

                )
                print("_____________________________________________________________________")
                numero += 1

    # OPCIÓN 8: REPORTE GENERAL DE INVENTARIO
    elif opcion == "8":
        if len(inventario) == 0:
            print("El inventario está vacío")
        else:
            print("Reporte de inventario")
            print("")

            numero = 1

            for nombre in inventario:
                cantidad = inventario[nombre]["cantidad"]
                costo_u = inventario[nombre]["costo_unitario"]
                precio_v = inventario[nombre]["precio_venta"]
                costo_lote = inventario[nombre]["costo_lote"]

                print(numero, ". Producto: ",
                      nombre,
                      " | Cantidad: ",
                      cantidad,
                      " | Costo unitario: S/",
                      round(costo_u, 2),
                      " | Precio unitario: S/",
                      round(precio_v, 2),
                      " | Costo lote: S/",
                      round(costo_lote, 2))
                print("______________________________________________________________________________________________________________________________")

                verificar_stock(cantidad)

                numero += 1

    # OPCIÓN 9: SALIR
    elif opcion == "9":
        print("Programa terminado")
        break

    else:
        print("Opción no válida. Seleccione un número del 1 al 9")


========== SISTEMA DE INVENTARIO ==========
1. Ingreso de mercadería
2. Consulta de stock por producto
3. Registrar venta
4. Consultar ventas por fecha
5. EERR simplificado
6. Producto más vendido
7. Historial de ventas
8. Reporte general de inventario
9. Salir
